# Session 4: Data Cleaning

**Learning goals:**

- Inspect a real dataset for missing values, duplicates, and data types.
- Handle missing values using removal and imputation strategies.
- Assess possible duplicates and remove them only when a reliable record key exists.
- Rename columns consistently.
- Use Python comparison operators and pandas Boolean masks to filter rows.

**Prerequisites:**

- Variables, lists, DataFrames, column selection, and basic functions.


## Outline

1. Setup and load the Titanic dataset
2. Inspect data quality
3. Handle missing values
4. Assess duplicates, drop columns, and rename columns
5. Filter data with comparison operations
6. Exercise, and common pitfalls


## 1. Setup and load the Titanic dataset

`sns.load_dataset("titanic")` downloads the example dataset the first time it is used. An internet connection may therefore be required on the first run.


In [35]:
# Import pandas for DataFrame operations.
import pandas as pd

# Import seaborn to access its built-in example datasets.
import seaborn as sns

In [36]:
# Load the real Titanic passenger dataset as a pandas DataFrame.
titanic = sns.load_dataset("titanic")

# Preview the first five rows to understand the columns and values.
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## 2. Inspect data quality

Before cleaning, inspect structure, missing values, duplicated rows, and suspicious categories. Always diagnose first; do not immediately delete every incomplete row.


In [37]:
# Print each column's non-missing count and data type.
titanic.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


In [38]:
# Count missing values in each column.
titanic.isna().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [39]:
# Count rows that are exact duplicates across every column.
titanic.duplicated().sum()

np.int64(107)

## 3. Handle missing values

Different columns need different decisions:

- `embarked`: only a very small number of rows are missing, so remove those rows.
- `age`: keep the rows and impute the median, which is less sensitive to extreme values than the mean.
- `deck`: most values are missing. Retain that information as the category `Unknown` rather than guessing a deck.
- `embark_town`: fill with the most frequent town. This also keeps the column consistent with the cleaned rows.

Keep the original `titanic` DataFrame unchanged and perform cleaning on a copy.


In [40]:
# Make an independent working copy so the raw dataset remains available.
clean_titanic = titanic.copy()

# Remove only rows whose embarkation code is missing.
clean_titanic = clean_titanic.dropna(subset=["embarked"]).copy()

# Compare row counts before and after removal.
print("Rows before:", len(titanic))
print("Rows after dropping missing embarked values:", len(clean_titanic))

Rows before: 891
Rows after dropping missing embarked values: 889


In [41]:
# Calculate the median from the available age values.
median_age = clean_titanic["age"].median()

# Replace missing ages with the median.
clean_titanic["age"] = clean_titanic["age"].fillna(median_age)

# Report the imputation value used, making the cleaning decision auditable.
print("Median age used for imputation:", median_age)

Median age used for imputation: 28.0


In [42]:
# Convert the categorical deck column to object before adding a new label.
clean_titanic["deck"] = clean_titanic["deck"].astype("object")

# Preserve missing-deck information with an explicit Unknown label.
clean_titanic["deck"] = clean_titanic["deck"].fillna("Unknown")

# Find the most frequently occurring embarkation town.
most_common_town = clean_titanic["embark_town"].mode().iloc[0]

# Replace missing town names with the mode.
clean_titanic["embark_town"] = clean_titanic["embark_town"].fillna(most_common_town)

In [43]:
# Verify that the selected missing-value treatments worked.
remaining_missing = clean_titanic.isna().sum()

# An empty Series means no missing values remain.
print("Total missing values remaining:", clean_titanic.isna().sum().sum())

Total missing values remaining: 0


## 4. Assess duplicates, drop columns, and rename columns

The Seaborn dataset has no passenger ID or name. Two rows with identical visible values could still represent different passengers, so `drop_duplicates()` must not be applied automatically. Diagnose possible duplicates first; remove them only when domain knowledge or a reliable business key confirms that they are duplicate records.


In [44]:
# Mark every row that shares all visible values with another row.
possible_duplicate_mask = clean_titanic.duplicated(keep=False)

# Count rows involved in possible duplicate groups.
print("Rows in possible duplicate groups:", possible_duplicate_mask.sum())

# Preview possible duplicates; do not delete them without a reliable passenger key.
possible_duplicates = clean_titanic.loc[possible_duplicate_mask].sort_values(
    ["pclass", "sex", "age", "fare"]
)
possible_duplicates.head(10)

Rows in possible duplicate groups: 163


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
369,1,1,female,24.0,0,0,69.3000,C,First,woman,False,B,Cherbourg,yes,True
641,1,1,female,24.0,0,0,69.3000,C,First,woman,False,B,Cherbourg,yes,True
64,0,1,male,28.0,0,0,27.7208,C,First,man,True,Unknown,Cherbourg,no,True
295,0,1,male,28.0,0,0,27.7208,C,First,man,True,Unknown,Cherbourg,no,True
443,1,2,female,28.0,0,0,13.0000,S,Second,woman,False,Unknown,Southampton,yes,True
635,1,2,female,28.0,0,0,13.0000,S,Second,woman,False,Unknown,Southampton,yes,True
53,1,2,female,29.0,1,0,26.0000,S,Second,woman,False,Unknown,Southampton,yes,False
133,1,2,female,29.0,1,0,26.0000,S,Second,woman,False,Unknown,Southampton,yes,False
458,1,2,female,50.0,0,0,10.5000,S,Second,woman,False,Unknown,Southampton,yes,True
526,1,2,female,50.0,0,0,10.5000,S,Second,woman,False,Unknown,Southampton,yes,True


In [45]:
# Drop 'alive' because it duplicates the information in numeric 'survived'.
clean_titanic = clean_titanic.drop(columns=["alive"])

# Rename selected columns with clearer analysis-friendly names.
clean_titanic = clean_titanic.rename(columns={
    "class": "passenger_class",
    "fare": "ticket_fare",
})

# Confirm the resulting column names.
clean_titanic.columns.tolist()

['survived',
 'pclass',
 'sex',
 'age',
 'sibsp',
 'parch',
 'ticket_fare',
 'embarked',
 'passenger_class',
 'who',
 'adult_male',
 'deck',
 'embark_town',
 'alone']

## 5. Filter data with comparison operations

### 5.1 Python comparison operators

A comparison asks a question and returns `True` or `False`.

| Operator | Meaning | Scalar example | Result |
|---|---|---|---|
| `==` | equal to | `3 == 3` | `True` |
| `!=` | not equal to | `3 != 5` | `True` |
| `>` | greater than | `8 > 5` | `True` |
| `>=` | greater than or equal to | `18 >= 18` | `True` |
| `<` | less than | `2 < 7` | `True` |
| `<=` | less than or equal to | `10 <= 10` | `True` |

For ordinary single Boolean values, Python uses `and`, `or`, and `not`. For pandas Series, use element-wise operators `&`, `|`, and `~`, and put every comparison inside parentheses.


In [46]:
# Scalar comparisons return one Boolean value each.
print("Equal:", 3 == 3)
print("Not equal:", 3 != 5)
print("Greater than:", 8 > 5)
print("Greater than or equal:", 18 >= 18)
print("Less than:", 2 < 7)
print("Less than or equal:", 10 <= 10)

# 'and' combines two ordinary Boolean expressions into one result.
print("Both scalar conditions are true:", (8 > 5) and (2 < 7))

Equal: True
Not equal: True
Greater than: True
Greater than or equal: True
Less than: True
Less than or equal: True
Both scalar conditions are true: True


### 5.2 Compare a pandas column

Comparing a Series with one value performs the operation row by row and returns a Boolean Series called a **mask**. The mask has the same index as the original DataFrame.


In [47]:
# Compare every age with 18; the result contains one Boolean per passenger.
adult_mask = clean_titanic["age"] >= 18

# Preview the first ten Boolean results.
print(adult_mask.head(10))

# Count True and False values in the mask.
print()
print("Mask counts:")
print(adult_mask.value_counts())

0     True
1     True
2     True
3     True
4     True
5     True
6     True
7    False
8     True
9    False
Name: age, dtype: bool

Mask counts:
age
True     776
False    113
Name: count, dtype: int64


In [48]:
# Pass the mask to .loc[] to keep only rows where the condition is True.
adults = clean_titanic.loc[adult_mask]

# Select a few useful columns to keep the output readable.
adults[["sex", "age", "passenger_class", "survived"]].head()

,sex,age,passenger_class,survived
0,male,22.0,Third,0
1,female,38.0,First,1
2,female,26.0,Third,1
3,female,35.0,First,1
4,male,35.0,Third,0


### 5.3 Combine conditions with AND: `&`

`&` keeps rows where **every** condition is `True`. Do not use Python's `and` between pandas Series because a Series contains many Boolean values, not one.


In [49]:
# Build separate masks so each condition can be inspected independently.
female_mask = clean_titanic["sex"] == "female"
adult_mask = clean_titanic["age"] >= 18

# Use &: passenger must be female AND at least 18 years old.
female_adult_mask = female_mask & adult_mask

# Filter rows and select relevant columns.
female_adults = clean_titanic.loc[
    female_adult_mask,
    ["sex", "age", "passenger_class", "ticket_fare", "survived"],
]
female_adults.head()

,sex,age,passenger_class,ticket_fare,survived
1,female,38.0,First,71.2833,1
2,female,26.0,Third,7.9250,1
3,female,35.0,First,53.1000,1
8,female,27.0,Third,11.1333,1
11,female,58.0,First,26.5500,1


### 5.4 Combine conditions with OR: `|`

`|` keeps rows where **at least one** condition is `True`. Parentheses are essential because comparisons and bitwise operators have different precedence rules.


In [50]:
# Keep passengers who were children OR travelled in first class.
child_or_first_class_mask = (
    (clean_titanic["age"] < 18)
    | (clean_titanic["passenger_class"] == "First")
)

# Apply the OR mask and preview the result.
child_or_first_class = clean_titanic.loc[
    child_or_first_class_mask,
    ["age", "passenger_class", "sex", "survived"],
]
child_or_first_class.head()

,age,passenger_class,sex,survived
1,38.0,First,female,1
3,35.0,First,female,1
6,54.0,First,male,0
7,2.0,Third,male,0
9,14.0,Second,female,1


### 5.5 Reverse a condition with NOT: `~`

`~` changes each `True` to `False` and each `False` to `True`. It is useful for excluding a category or reversing an existing mask.


In [51]:
# Test whether each passenger embarked at Southampton.
southampton_mask = clean_titanic["embark_town"] == "Southampton"

# Reverse the mask to keep passengers who did not embark at Southampton.
not_southampton = clean_titanic.loc[
    ~southampton_mask,
    ["embark_town", "passenger_class", "ticket_fare"],
]
not_southampton.head()

,embark_town,passenger_class,ticket_fare
1,Cherbourg,First,71.2833
5,Queenstown,Third,8.4583
9,Cherbourg,Second,30.0708
16,Queenstown,Third,29.1250
19,Cherbourg,Third,7.2250


### 5.6 Match several values with `.isin()`

Repeated equality checks such as `(x == "First") | (x == "Second")` work, but `.isin([...])` is shorter and easier to extend.


In [52]:
# Return True for First- or Second-class passengers.
upper_class_mask = clean_titanic["passenger_class"].isin(["First", "Second"])

# Apply the membership mask.
upper_class_passengers = clean_titanic.loc[
    upper_class_mask,
    ["passenger_class", "age", "ticket_fare", "survived"],
]
upper_class_passengers.head()

,passenger_class,age,ticket_fare,survived
1,First,38.0,71.2833,1
3,First,35.0,53.1000,1
6,First,54.0,51.8625,0
9,Second,14.0,30.0708,1
11,First,58.0,26.5500,1


### 5.7 Filter a numeric range with `.between()`

For a pandas Series, do not write `18 <= series <= 40`. Use two comparisons joined by `&`, or use `.between(18, 40)`. By default, both endpoints are included.


In [53]:
# Keep ages from 18 through 40, including both boundary values.
age_18_to_40_mask = clean_titanic["age"].between(18, 40)

# Apply the range mask and show key columns.
passengers_age_18_to_40 = clean_titanic.loc[
    age_18_to_40_mask,
    ["age", "sex", "passenger_class", "survived"],
]
passengers_age_18_to_40.head()

,age,sex,passenger_class,survived
0,22.0,male,Third,0
1,38.0,female,First,1
2,26.0,female,Third,1
3,35.0,female,First,1
4,35.0,male,Third,0


### 5.8 Build a practical multi-condition filter

Complex filters are easier to read when each business rule is named. The final mask below keeps passengers who:

- embarked at Cherbourg or Queenstown
- were aged 18 to 40 inclusive
- paid more than 20
- did not travel in Third class


In [54]:
# Create one named mask for each filtering rule.
selected_ports_mask = clean_titanic["embarked"].isin(["C", "Q"])
working_age_mask = clean_titanic["age"].between(18, 40)
higher_fare_mask = clean_titanic["ticket_fare"] > 20
not_third_class_mask = clean_titanic["passenger_class"] != "Third"

# Combine all rules with &: every rule must be True.
analysis_mask = (
    selected_ports_mask
    & working_age_mask
    & higher_fare_mask
    & not_third_class_mask
)

# Filter rows and return only the columns needed for analysis.
analysis_table = clean_titanic.loc[
    analysis_mask,
    [
        "survived",
        "passenger_class",
        "sex",
        "age",
        "ticket_fare",
        "embark_town",
    ],
]

# Reset the index so the filtered table has consecutive row labels.
analysis_table = analysis_table.reset_index(drop=True)
analysis_table.head(10)

,survived,passenger_class,sex,age,ticket_fare,embark_town
0,1,First,female,38.0,71.2833,Cherbourg
1,0,First,male,40.0,27.7208,Cherbourg
2,1,First,female,28.0,146.5208,Cherbourg
3,0,First,male,28.0,82.1708,Cherbourg
4,0,First,male,28.0,27.7208,Cherbourg
5,1,First,male,23.0,63.3583,Cherbourg
6,0,First,male,24.0,247.5208,Cherbourg
7,0,Second,male,32.5,30.0708,Cherbourg
8,0,First,male,24.0,79.2000,Cherbourg
9,1,First,male,40.0,31.0000,Cherbourg


In [55]:
# Summarise how strongly the filter reduced the dataset.
print("Rows in cleaned dataset:", len(clean_titanic))
print("Rows after practical filter:", len(analysis_table))

Rows in cleaned dataset: 889
Rows after practical filter: 61


## 6. Exercise

Create a table called `surviving_families` containing passengers who:

1. survived (`survived == 1`)
2. travelled with at least one parent/child **or** at least one sibling/spouse
3. paid a fare of 15 or more
4. display only `passenger_class`, `sex`, `age`, `sibsp`, `parch`, and `ticket_fare`

Remember to place parentheses around the two family comparisons before joining them with `|`.


In [ ]:
# TODO: complete each condition, then combine the masks.
survived_mask = clean_titanic["survived"] == 1
family_mask = (clean_titanic["parch"] > 0) | (clean_titanic["sibsp"] > 0)
fare_mask = clean_titanic["ticket_fare"] >= 15

# TODO: combine the three masks with & and select the requested columns.
surviving_families = clean_titanic.loc[
    survived_mask & family_mask & fare_mask,
    ["passenger_class", "sex", "age", "sibsp", "parch", "ticket_fare"],
]
surviving_families.head()

## Common pitfalls

**Common pitfalls**

- Use `==` for comparison; `=` assigns a value.
- Use `&`, `|`, and `~` with pandas Series—not `and`, `or`, and `not`.
- Put each pandas comparison inside parentheses when combining conditions.
- Comparisons involving missing numeric values usually return `False`; decide how to handle missing values before filtering.
- Avoid `dropna()` with no arguments unless removing every row with any missing value is truly intended.
- Use `.copy()` when creating a DataFrame that you plan to modify.
